In [1]:
import json
import pinecone
import requests
from tqdm import tqdm
import time
from typing import List, Dict, Any
from dotenv import load_dotenv
import os


In [2]:
load_dotenv()
PINECONE_API = os.getenv("PINECONE_API")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [3]:
import json


def prepare_rules_for_indexing(json_file):

    with open(json_file, "r", encoding="utf-8") as f:
        laws = json.load(f)

    records = []

    for law in laws:

        metadata = law["metadata"]

        for idx, rule in enumerate(
            law["rules_extracted"]
        ):

            record = {
    "id": f"{metadata['law_number']}_{idx}",

    "text": rule["condition"],

    "metadata": {
        "law_number": metadata["law_number"],
        "law_name": metadata["law_name"],
        "article": metadata.get("article"),
        "paragraph": metadata.get("paragraph"),
        "jurisdiction": metadata.get("jurisdiction"),
        "rule_type": rule["rule_type"],
        "applies_to": ",".join(rule["applies_to"]),
        "priority": metadata.get("priority")
    }

            }

            records.append(record)

    return records

In [27]:
records = prepare_rules_for_indexing(
    "./processed_laws.json"
)

records

[{'id': '4738/2020_0',
  'text': 'If a debtor is categorized as medium or high insolvency risk through the electronic early warning system and is a natural person without business income, they are entitled to free advisory and budgeting services from Debt Servicing Centers[cite: 1].',
  'metadata': {'law_number': '4738/2020',
   'law_name': 'Settlement of Debts and Provision of a Second Chance',
   'article': '1-12',
   'paragraph': 'Multiple',
   'jurisdiction': 'Greece',
   'rule_type': 'consumer_protection',
   'applies_to': 'retail_borrowers,individuals',
   'priority': 'HIGH'}},
 {'id': '4738/2020_1',
  'text': 'A debtor is ineligible for out-of-court settlement if 90% or more of their total debts across financial institutions, the State, and Social Security Agencies are owed to a single financial institution[cite: 1].',
  'metadata': {'law_number': '4738/2020',
   'law_name': 'Settlement of Debts and Provision of a Second Chance',
   'article': '1-12',
   'paragraph': 'Multiple',

In [ ]:
import json
import pinecone
import requests
from tqdm import tqdm
import time
from typing import List, Dict, Any

# ============================================
# 1. OPENROUTER EMBEDDING FUNCTION
# ============================================
def get_embeddings(texts: List[str], api_key: str, model: str = "openai/text-embedding-3-small") -> List[List[float]]:
    """
    Get embeddings from OpenRouter's NVIDIA model.
    
    Args:
        texts: List of texts to embed
        api_key: OpenRouter API key
        model: Model name
    
    Returns:
        List of embedding vectors
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "input": texts
    }
    
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        json=payload,
        headers=headers
    )
    response.raise_for_status()
    
    data = response.json()
    embeddings = [item['embedding'] for item in data['data']]
    
    return embeddings

# ============================================
# 2. CREATE SEARCHABLE TEXT FOR EMBEDDING
# ============================================
def create_embedding_text(chunk: Dict[str, Any]) -> str:
    """Combine relevant fields into a single searchable text."""
    rules_text = '; '.join([r['condition'] for r in chunk.get('rules_extracted', [])])
    keywords = ', '.join(chunk.get('metadata', {}).get('keywords', []))
    applies_to = ', '.join(chunk.get('rules_extracted', [{}])[0].get('applies_to', [])) if chunk.get('rules_extracted') else 'N/A'
    
    return f"""
    Law: {chunk.get('metadata', {}).get('law_name', 'Unknown')} 
    Article: {chunk.get('metadata', {}).get('article', 'Unknown')}
    Summary: {chunk.get('summary', '')}
    Rules: {rules_text}
    Keywords: {keywords}
    Applies to: {applies_to}
    """

# ============================================
# 3. INGEST JSON INTO PINECONE
# ============================================
from pinecone import Pinecone, ServerlessSpec

def ingest_laws_to_pinecone(
    json_file_path: str,
    index_name: str = "sailok-laws",
    api_key: str = "YOUR_PINECONE_API_KEY",
    openrouter_key: str = "YOUR_OPENROUTER_KEY",
    batch_size: int = 100,
    cloud: str = "aws",
    region: str = "us-east-1",
):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        law_chunks = json.load(f)

    print(f"📄 Loaded {len(law_chunks)} law chunks")

    pc = Pinecone(api_key=api_key)

    # Probe the embedding model to find its actual output dimension
    sample_embedding = get_embeddings(["dimension probe"], openrouter_key)[0]
    dimension = len(sample_embedding)
    print(f"📐 Detected embedding dimension: {dimension}")

    existing_indexes = [idx["name"] for idx in pc.list_indexes()]
    if index_name in existing_indexes:
        pc.delete_index(index_name)
        print(f"🗑️  Deleted existing index: {index_name}")

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine',
        spec=ServerlessSpec(cloud=cloud, region=region)
    )
    print(f"✅ Created index: {index_name} (dim={dimension})")

    index = pc.Index(index_name)
    # Prepare batch
    vectors_to_upsert = []
    
    # Process chunks
    for i, chunk in enumerate(tqdm(law_chunks, desc="Processing chunks")):
        # Generate unique ID
        doc_id = f"{chunk.get('metadata', {}).get('law_name', 'unknown')}_{chunk.get('metadata', {}).get('article', 'unknown')}_{i}"
        
        # Create searchable text for embedding
        search_text = create_embedding_text(chunk)
        
        # Generate embedding (with rate limit handling)
        try:
            embedding = get_embeddings([search_text], openrouter_key)[0]
        except Exception as e:
            print(f"⚠️  Error embedding chunk {i}: {e}")
            time.sleep(5)  # Wait and retry
            embedding = get_embeddings([search_text], openrouter_key)[0]
        
        # Prepare metadata (what you'll filter on)
        metadata = {
            "law_name": chunk.get('metadata', {}).get('law_name', ''),
            "law_number": chunk.get('metadata', {}).get('law_number', ''),
            "article": chunk.get('metadata', {}).get('article', ''),
            "paragraph": chunk.get('metadata', {}).get('paragraph', ''),
            "priority": chunk.get('metadata', {}).get('priority', 'MEDIUM'),
            "confidence": chunk.get('confidence', 0.0),
            "summary": chunk.get('summary', ''),  # Truncate for storage
            "text_preview": chunk.get('text', ''),  # Truncate for storage
            "rules_count": len(chunk.get('rules_extracted', [])),
            "keywords": ', '.join(chunk.get('metadata', {}).get('keywords', []))
        }
        
        # Add to batch
        vectors_to_upsert.append((doc_id, embedding, metadata))
        
        # When batch is full, upsert to Pinecone
        if len(vectors_to_upsert) >= batch_size:
            index.upsert(vectors=vectors_to_upsert)
            print(f"✅ Upserted batch of {len(vectors_to_upsert)} chunks")
            vectors_to_upsert = []
            time.sleep(1)  # Rate limit for free API
    
    # Upsert remaining
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
        print(f"✅ Upserted final batch of {len(vectors_to_upsert)} chunks")
    
    print(f"\n🎉 SUCCESS: Ingested {len(law_chunks)} law chunks into Pinecone")
    print(f"📊 Index: {index_name}")
    print(f"📊 Total vectors: {index.describe_index_stats()['total_vector_count']}")
    
    return index

# ============================================
# 4. RUN THE INGESTION
# ============================================
if __name__ == "__main__":
    index = ingest_laws_to_pinecone(
        json_file_path="processed_laws.json",  # Your JSON file
        index_name="sailok-laws",
        api_key=PINECONE_API,  # Replace with your key
        # environment="us-west1-gcp",  # Replace with your environment
        openrouter_key=OPENROUTER_API_KEY,  # Replace with your key
        batch_size=100,
        # dimension=768  # NVIDIA embedding dimension
    )

📄 Loaded 9 law chunks
📐 Detected embedding dimension: 2048
🗑️  Deleted existing index: sailok-laws
✅ Created index: sailok-laws (dim=2048)


Processing chunks: 100%|██████████| 9/9 [00:07<00:00,  1.19it/s]


✅ Upserted final batch of 9 chunks

🎉 SUCCESS: Ingested 9 law chunks into Pinecone
📊 Index: sailok-laws
📊 Total vectors: 9


In [25]:
def search_and_show_full_metadata(query, top_k=3):
    """Search and display full metadata for results."""
    
    # Get embedding for query
    query_embedding = get_embeddings([query], OPENROUTER_API_KEY)[0]
    
    # Search
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True  # Critical: ensures metadata is returned
    )
    
    print(f"\n🔍 Query: {query}")
    print("=" * 60)
    
    for i, match in enumerate(results['matches'], 1):
        print(f"\n📜 Result {i} (Score: {match['score']:.2f})")
        print(f"   ID: {match['id']}")
        print("-" * 40)
        
        # Show all metadata
        metadata = match['metadata']
        for key, value in metadata.items():
            if key in ['summary', 'text_preview']:
                print(f"\n   {key.upper()}:")
                print(f"   {value}")  # Full text
            elif key == 'keywords':
                print(f"   {key}: {value[:100]}...")  # Truncate for display
            else:
                print(f"   {key}: {value}")
    
    return results

# Test it
search_and_show_full_metadata("Can a 17 year old get a loan?", top_k=2)


🔍 Query: Can a 17 year old get a loan?

📜 Result 1 (Score: 0.16)
   ID: Code of Conduct under Law 4224/2013_Sections A-K, Annexes 1-3_2
----------------------------------------
   article: Sections A-K, Annexes 1-3
   confidence: 0.98
   keywords: cooperating borrower, loan arrears, forbearance, Code of Conduct, non-performing loans, Arrears Reso...
   law_name: Code of Conduct under Law 4224/2013
   law_number: Decision 116/25.08.2014
   paragraph: Multiple
   priority: HIGH
   rules_count: 9

   SUMMARY:
   This decision establishes the Greek Code of Conduct for managing loan arrears, setting out strict timelines and procedures for banks to offer restructuring options and defining the obligations of a 'cooperating borrower.'[cite: 2]

   TEXT_PREVIEW:
   Bank of Greece Credit and Insurance Committee Decision No. 116/25.08.2014 introduces a Code of Conduct under Law 4224/2013 regarding the management of non-performing loans and loans in arrears[cite: 2]. The Code establishes procedur

QueryResponse(matches=[ScoredVector(id='Code of Conduct under Law 4224/2013_Sections A-K, Annexes 1-3_2', score=0.159448236, values=[], metadata={'article': 'Sections A-K, Annexes 1-3', 'confidence': 0.98, 'keywords': 'cooperating borrower, loan arrears, forbearance, Code of Conduct, non-performing loans, Arrears Resolution Procedure, Bank of Greece, Standardised Statement of Financial Information', 'law_name': 'Code of Conduct under Law 4224/2013', 'law_number': 'Decision 116/25.08.2014', 'paragraph': 'Multiple', 'priority': 'HIGH', 'rules_count': 9, 'summary': "This decision establishes the Greek Code of Conduct for managing loan arrears, setting out strict timelines and procedures for banks to offer restructuring options and defining the obligations of a 'cooperating borrower.'[cite: 2]", 'text_preview': "Bank of Greece Credit and Insurance Committee Decision No. 116/25.08.2014 introduces a Code of Conduct under Law 4224/2013 regarding the management of non-performing loans and loan

# NEW DATA


In [29]:
import json
import time
import requests

from typing import List, Dict, Any

from pinecone import Pinecone, ServerlessSpec
from tqdm import tqdm


# =====================================================
# EMBEDDINGS
# =====================================================

def get_embeddings(
    texts: List[str],
    api_key: str,
    model: str = "openai/text-embedding-3-small"
) -> List[List[float]]:

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "input": texts
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        json=payload,
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    return [
        item["embedding"]
        for item in data["data"]
    ]


# =====================================================
# CONVERT LAW JSON -> RULE RECORDS
# =====================================================

def prepare_rule_records(
    laws: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:

    records = []

    for law in laws:

        metadata = law.get("metadata", {})

        law_number = metadata.get(
            "law_number",
            "unknown"
        )

        law_name = metadata.get(
            "law_name",
            "unknown"
        )

        article = metadata.get(
            "article",
            ""
        )

        paragraph = metadata.get(
            "paragraph",
            ""
        )

        jurisdiction = metadata.get(
            "jurisdiction",
            ""
        )

        priority = metadata.get(
            "priority",
            "MEDIUM"
        )

        summary = law.get(
            "summary",
            ""
        )

        confidence = law.get(
            "confidence",
            0.0
        )

        rules = law.get(
            "rules_extracted",
            []
        )

        for idx, rule in enumerate(rules):

            condition = rule.get(
                "condition",
                ""
            ).strip()

            if not condition:
                continue

            records.append(
                {
                    "id":
                        f"{law_number}_rule_{idx}",

                    "text":
                        condition,

                    "metadata": {

                        # rule info
                        "condition":
                            condition,

                        "rule_type":
                            rule.get(
                                "rule_type",
                                ""
                            ),

                        "value":
                            str(
                                rule.get(
                                    "value",
                                    ""
                                )
                            ),

                        "applies_to":
                            ",".join(
                                rule.get(
                                    "applies_to",
                                    []
                                )
                            ),

                        # citation
                        "law_number":
                            law_number,

                        "law_name":
                            law_name,

                        "article":
                            article,

                        "paragraph":
                            paragraph,

                        # legal metadata
                        "jurisdiction":
                            jurisdiction,

                        "priority":
                            priority,

                        "confidence":
                            float(confidence),

                        "summary":
                            summary[:1500]
                    }
                }
            )

    return records


# =====================================================
# INGEST
# =====================================================

def ingest_rules_to_pinecone(
    json_file_path: str,
    index_name: str,
    pinecone_api_key: str,
    openrouter_key: str,
    batch_size: int = 50,
    cloud: str = "aws",
    region: str = "us-east-1"
):

    print("\nLoading laws...")

    with open(
        json_file_path,
        "r",
        encoding="utf-8"
    ) as f:

        laws = json.load(f)

    print(
        f"Loaded {len(laws)} laws"
    )

    records = prepare_rule_records(
        laws
    )

    print(
        f"Prepared {len(records)} rule records"
    )

    # -------------------------------------------------
    # Pinecone setup
    # -------------------------------------------------

    pc = Pinecone(
        api_key=pinecone_api_key
    )

    sample_embedding = get_embeddings(
        ["dimension probe"],
        openrouter_key
    )[0]

    dimension = len(
        sample_embedding
    )

    print(
        f"Embedding dimension: {dimension}"
    )

    existing_indexes = [
        idx["name"]
        for idx in pc.list_indexes()
    ]

    if index_name in existing_indexes:

        print(
            f"Deleting existing index: {index_name}"
        )

        pc.delete_index(index_name)

        time.sleep(5)

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )

    print(
        f"Created index: {index_name}"
    )

    index = pc.Index(index_name)

    # -------------------------------------------------
    # Build vectors
    # -------------------------------------------------

    vectors = []

    for record in tqdm(
        records,
        desc="Embedding rules"
    ):

        try:

            embedding = get_embeddings(
                [record["text"]],
                openrouter_key
            )[0]

        except Exception as e:

            print(
                f"Embedding error: {e}"
            )

            time.sleep(5)

            embedding = get_embeddings(
                [record["text"]],
                openrouter_key
            )[0]

        vectors.append(
            (
                record["id"],
                embedding,
                record["metadata"]
            )
        )

    print(
        f"Generated {len(vectors)} vectors"
    )

    # -------------------------------------------------
    # Upsert
    # -------------------------------------------------

    for i in range(
        0,
        len(vectors),
        batch_size
    ):

        batch = vectors[
            i:i + batch_size
        ]

        index.upsert(
            vectors=batch
        )

        print(
            f"Upserted "
            f"{min(i + batch_size, len(vectors))}"
            f"/{len(vectors)}"
        )

        time.sleep(1)

    stats = index.describe_index_stats()

    print("\nSUCCESS")
    print(
        f"Index: {index_name}"
    )
    print(
        f"Total vectors: "
        f"{stats['total_vector_count']}"
    )

    return index


# =====================================================
# RUN
# =====================================================

if __name__ == "__main__":

    ingest_rules_to_pinecone(
        json_file_path="processed_laws.json",
        index_name="sailok-laws",
        pinecone_api_key=PINECONE_API,
        openrouter_key=OPENROUTER_API_KEY,
        batch_size=50
    )


Loading laws...
Loaded 9 laws
Prepared 57 rule records
Embedding dimension: 1536
Created index: sailok-laws


Embedding rules: 100%|██████████| 57/57 [01:02<00:00,  1.09s/it]


Generated 57 vectors
Upserted 50/57
Upserted 57/57

SUCCESS
Index: sailok-laws
Total vectors: 57


In [31]:
with open(
    './processed_laws.json',
    "r",
    encoding="utf-8"
) as f:

    laws = json.load(f)

print(
    f"Loaded {len(laws)} laws"
)

records = prepare_rule_records(
    laws
)


Loaded 9 laws


In [32]:
records

[{'id': '4738/2020_rule_0',
  'text': 'If a debtor is categorized as medium or high insolvency risk through the electronic early warning system and is a natural person without business income, they are entitled to free advisory and budgeting services from Debt Servicing Centers[cite: 1].',
  'metadata': {'condition': 'If a debtor is categorized as medium or high insolvency risk through the electronic early warning system and is a natural person without business income, they are entitled to free advisory and budgeting services from Debt Servicing Centers[cite: 1].',
   'rule_type': 'consumer_protection',
   'value': 'None',
   'applies_to': 'retail_borrowers,individuals',
   'law_number': '4738/2020',
   'law_name': 'Settlement of Debts and Provision of a Second Chance',
   'article': '1-12',
   'paragraph': 'Multiple',
   'jurisdiction': 'Greece',
   'priority': 'HIGH',
   'confidence': 0.98,
   'summary': 'This law establishes the framework for insolvency prevention and the Out-of-Cou

# Dense + BM25

In [6]:
import json
import time
import pickle
import requests

from typing import List, Dict, Any

from pinecone import Pinecone, ServerlessSpec
from rank_bm25 import BM25Okapi
from tqdm import tqdm


# =====================================================
# EMBEDDINGS
# =====================================================

def get_embeddings(
    texts: List[str],
    api_key: str,
    model: str = "openai/text-embedding-3-small"
):

    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "input": texts
        },
        timeout=60
    )

    response.raise_for_status()

    return [
        item["embedding"]
        for item in response.json()["data"]
    ]


# =====================================================
# PREPARE RULE RECORDS
# =====================================================

def prepare_rule_records(
    laws: List[Dict[str, Any]]
):

    records = []

    for law in laws:

        metadata = law.get("metadata", {})

        for idx, rule in enumerate(
            law.get("rules_extracted", [])
        ):

            condition = rule.get(
                "condition",
                ""
            ).strip()

            if not condition:
                continue

            record = {

                "id":
                    f"{metadata.get('law_number','unknown')}_{idx}",

                "text":
                    condition,

                "metadata": {

                    # embedded text
                    "condition":
                        condition,

                    # rule metadata
                    "rule_type":
                        rule.get(
                            "rule_type",
                            ""
                        ),

                    "value":
                        str(
                            rule.get(
                                "value",
                                ""
                            )
                        ),

                    "applies_to":
                        ",".join(
                            rule.get(
                                "applies_to",
                                []
                            )
                        ),

                    # citation metadata
                    "law_number":
                        metadata.get(
                            "law_number",
                            ""
                        ),

                    "law_name":
                        metadata.get(
                            "law_name",
                            ""
                        ),

                    "article":
                        metadata.get(
                            "article",
                            ""
                        ),

                    "paragraph":
                        metadata.get(
                            "paragraph",
                            ""
                        ),

                    # additional filters
                    "jurisdiction":
                        metadata.get(
                            "jurisdiction",
                            ""
                        ),

                    "priority":
                        metadata.get(
                            "priority",
                            "MEDIUM"
                        ),

                    "confidence":
                        float(
                            law.get(
                                "confidence",
                                0.0
                            )
                        ),

                    "summary":
                        law.get(
                            "summary",
                            ""
                        )[:1500]
                }
            }

            records.append(record)

    return records


# =====================================================
# BM25 INDEX
# =====================================================

def build_bm25_index(
    records,
    output_file="bm25_rules.pkl"
):

    corpus = [
        r["text"]
        for r in records
    ]

    tokenized = [
        doc.lower().split()
        for doc in corpus
    ]

    bm25 = BM25Okapi(tokenized)

    bm25_store = {
        "bm25": bm25,
        "records": records
    }

    with open(
        output_file,
        "wb"
    ) as f:

        pickle.dump(
            bm25_store,
            f
        )

    print(
        f"✅ BM25 saved to {output_file}"
    )


# =====================================================
# PINECONE INGESTION
# =====================================================

def ingest_rules_to_pinecone(
    json_file_path: str,
    pinecone_api_key: str,
    openrouter_key: str,
    index_name: str = "sailok-laws",
    batch_size: int = 50,
    cloud: str = "aws",
    region: str = "us-east-1",
    bm25_output: str = "bm25_rules.pkl"
):

    # ------------------------------------------
    # Load JSON
    # ------------------------------------------

    with open(
        json_file_path,
        "r",
        encoding="utf-8"
    ) as f:

        laws = json.load(f)

    print(
        f"Loaded {len(laws)} laws"
    )

    records = prepare_rule_records(
        laws
    )

    print(
        f"Prepared {len(records)} rule records"
    )

    # ------------------------------------------
    # Build BM25
    # ------------------------------------------

    build_bm25_index(
        records,
        bm25_output
    )

    # ------------------------------------------
    # Pinecone Setup
    # ------------------------------------------

    pc = Pinecone(
        api_key=pinecone_api_key
    )

    probe = get_embeddings(
        ["dimension probe"],
        openrouter_key
    )[0]

    dimension = len(probe)

    print(
        f"Embedding dimension = {dimension}"
    )

    existing = [
        idx["name"]
        for idx in pc.list_indexes()
    ]

    if index_name in existing:

        pc.delete_index(
            index_name
        )

        print(
            f"Deleted existing index {index_name}"
        )

        time.sleep(5)

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )

    print(
        f"Created index {index_name}"
    )

    index = pc.Index(index_name)

    # ------------------------------------------
    # Embed Rules
    # ------------------------------------------

    vectors = []

    for record in tqdm(
        records,
        desc="Embedding Rules"
    ):

        try:

            embedding = get_embeddings(
                [record["text"]],
                openrouter_key
            )[0]

        except Exception:

            time.sleep(5)

            embedding = get_embeddings(
                [record["text"]],
                openrouter_key
            )[0]

        vectors.append(
            (
                record["id"],
                embedding,
                record["metadata"]
            )
        )

    # ------------------------------------------
    # Upsert
    # ------------------------------------------

    for i in range(
        0,
        len(vectors),
        batch_size
    ):

        batch = vectors[
            i:i + batch_size
        ]

        index.upsert(
            vectors=batch
        )

        print(
            f"Upserted "
            f"{min(i+batch_size,len(vectors))}"
            f"/{len(vectors)}"
        )

        time.sleep(1)

    stats = index.describe_index_stats()

    print("\n🎉 DONE")
    print(
        f"Total vectors = "
        f"{stats['total_vector_count']}"
    )

    return index

In [9]:

if __name__ == "__main__":

    index = ingest_rules_to_pinecone(
        json_file_path="processed_laws.json",
        index_name="sailok-laws",
        pinecone_api_key=PINECONE_API,
        openrouter_key=OPENROUTER_API_KEY,
        batch_size=50
    )

Loaded 9 laws
Prepared 57 rule records
✅ BM25 saved to bm25_rules.pkl
Embedding dimension = 1536
Deleted existing index sailok-laws
Created index sailok-laws


Embedding Rules: 100%|██████████| 57/57 [02:20<00:00,  2.47s/it]


Upserted 50/57
Upserted 57/57

🎉 DONE
Total vectors = 57


In [22]:
query = "Can a 17 year old get a mortgage in Greece?"

query_embedding = get_embeddings(
    [query],
    OPENROUTER_API_KEY
)[0]


results = index.query(
    vector=query_embedding,
    top_k=10,
    include_metadata=True
)

In [21]:
results

QueryResponse(matches=[ScoredVector(id='2021/2167_3', score=0.434686273, values=[], metadata={'applies_to': 'retail_borrowers,mortgages,consumer_loans,banks,credit_purchasers', 'article': '1-17', 'condition': 'Creditors must have adequate policies and exercise reasonable forbearance (e.g., term extension, interest rate change, payment holiday, partial debt forgiveness) before initiating foreclosure proceedings against a consumer.', 'confidence': 0.98, 'jurisdiction': 'European Union', 'law_name': 'Directive (EU) 2021/2167 on credit servicers and credit purchasers', 'law_number': '2021/2167', 'paragraph': 'Multiple', 'priority': 'HIGH', 'rule_type': 'repayment_limit', 'summary': 'Establishes an EU-wide framework for entities purchasing or servicing non-performing bank loans. It ensures that when a bank sells a non-performing loan to a credit purchaser or assigns it to a credit servicer, consumer protections, original contract rights, and fair collection standards are fully preserved.', 

In [17]:
def dense_search(
    query: str,
    index,
    openrouter_key: str,
    top_k: int = 4
):

    query_embedding = get_embeddings(
        [query],
        openrouter_key
    )[0]

    return index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

results = dense_search(
    query="Can a 17 year old get a mortgage in Greece? as a Pakistani Citizenship",
    index=index,
    openrouter_key=OPENROUTER_API_KEY
)

for match in results["matches"]:
    print(match["score"])
    print(match["metadata"]["condition"])
    print(match["metadata"]["law_number"])
    print()

0.475646794
Credit purchasers domiciled outside the EU must designate a representative based in the EU who assumes equal legal compliance obligations before the Bank of Greece[cite: 1].
5072/2023

0.444206148
If an individual is physically present in Greece for more than 183 days within any twelve-month period (continuously or intermittently), then they are deemed a tax resident of Greece for the tax year in which the 12-month period ends.
4172/2013

0.39059332
Credit servicing firms (E.D.A.D.P.) must be established as sociétés anonymes in Greece with a minimum paid-up share capital of €100,000 deposited in an un-affiliated credit institution[cite: 1].
5072/2023

0.388045192
The Bank of Greece must assess licensing application completeness within 45 days, allow a minimum of 15 days for missing data submission, and render a final approval or rejection decision within 90 days of complete application (or 60 days of missing data submission), with silence constituting implicit rejection[cit

In [24]:
import pickle

with open("bm25_rules.pkl", "rb") as f:
    store = pickle.load(f)

bm25 = store["bm25"]
records = store["records"]

EOFError: Ran out of input

In [23]:
with open("bm25_rules.pkl", "wb") as f:
    pickle.dump(
        {
            "bm25": bm25,
            "records": records
        },
        f
    )

NameError: name 'bm25' is not defined

In [ ]:
import pickle
import numpy as np

with open("bm25_rules.pkl", "rb") as f:
    store = pickle.load(f)

bm25 = store["bm25"]
records = store["records"]

query = "mortgage age requirement"

query_tokens = query.lower().split()

scores = bm25.get_scores(query_tokens)

top_k = 10

top_indices = np.argsort(scores)[::-1][:top_k]

results = []

for idx in top_indices:

    if scores[idx] <= 0:
        continue

    results.append({
        "score": float(scores[idx]),
        "record": records[idx]
    })

for r in results:
    print("=" * 80)
    print("Score:", r["score"])
    print("Condition:", r["record"]["text"])
    print("Citation:")
    print(
        f"{r['record']['metadata']['law_name']} "
        f"({r['record']['metadata']['law_number']}) "
        f"Article {r['record']['metadata']['article']}"
    )